In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('Acetylene production')

In [4]:
bd.databases

Databases dictionary with 10 object(s):
	acetylene-production-Base-2020
	acetylene-production-Base-2050
	acetylene-production-RCP19-2050
	acetylene-production-RCP26-2050
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-Base-2020
	ecoinvent-3.10-cutoff-Base-2050
	ecoinvent-3.10-cutoff-RCP19-2050
	ecoinvent-3.10-cutoff-RCP26-2050

In [5]:
# ei_db = wurst.extract_brightway2_databases('ecoinvent-3.10-cutoff-Base-2020')

In [6]:
# for ds in ei_db:
#     if 'biomethane production, from biogas upgrading, using amine scrubbing' in ds['name'] and 'World' in ds['location']:
#         print(ds['reference product'], ds['name'], ds['location'], ds['unit'])
#         act = ds

In [7]:
# import inventories from excel
excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories_no_allocation.xlsx'))
# excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories.xlsx'))
excel_import.apply_strategies()
excel_import.match_database('ecoinvent-3.10-cutoff-Base-2020', fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excel_import.match_database('ecoinvent-3.10-biosphere', fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excel_import.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from new database
excel_import.statistics()

Extracted 7 worksheets in 0.05 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 9.51 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
29 datasets
297 exchanges
0 unlinked exchang

(29, 297, 0)

In [8]:
excel_import.write_database()

100%|██████████| 29/29 [00:00<00:00, 2611.59it/s]


Vacuuming database 
Created database: acetylene-production-Base-2020


In [ ]:
# import inventories from excel
excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories_no_allocation_base.xlsx'))
# excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories.xlsx'))
excel_import.apply_strategies()
excel_import.match_database('ecoinvent-3.10-cutoff', fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excel_import.match_database('ecoinvent-3.10-cutoff-Base-2020', fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excel_import.match_database('ecoinvent-3.10-biosphere', fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excel_import.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from new database
excel_import.statistics()

Extracted 6 worksheets in 0.04 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 9.41 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
2

100%|██████████| 25/25 [00:00<00:00, 1670.77it/s]


Vacuuming database 
Created database: acetylene-production


In [9]:
act = [ds for ds in bd.Database('acetylene-production-Base-2020') if 'vinyl chloride monomer production, bio acetylene' == ds['name'] and 'kilogram' in ds['unit']][0]
act

'vinyl chloride monomer production, bio acetylene' (kilogram, GLO, None)

In [10]:
method = ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")

In [11]:
lca = bc.LCA({act: 1}, method = method)
lca.lci()
lca.lcia()
lca.score

1.6772151546116922

In [12]:
database_names = bd.databases
ecoinvent_db_names = []
premise_db_names = []
for db_name in database_names:
    if '2050' in db_name and 'ecoinvent' in db_name:
        ecoinvent_db_names.append(db_name)
        premise_db_names.append(db_name.replace('ecoinvent-3.10-cutoff-', ''))
ecoinvent_db_names.sort()
premise_db_names.sort()

In [13]:
ecoinvent_db_names

['ecoinvent-3.10-cutoff-Base-2050',
 'ecoinvent-3.10-cutoff-RCP19-2050',
 'ecoinvent-3.10-cutoff-RCP26-2050']

In [14]:
premise_db_names

['Base-2050', 'RCP19-2050', 'RCP26-2050']

In [15]:
biosphere_db = bd.Database('ecoinvent-3.10-biosphere') # import the biosphere database
biosphere_db_dict = [ds.as_dict() for ds in biosphere_db] # convert biosphere database to dictionary # maybe not needed?

In [ ]:
base_db = wurst.extract_brightway2_databases('acetylene-production-Base-2020')

In [17]:
for premise_db_name in premise_db_names:
    print('linking database ' + premise_db_name + '...')
    premise_db_dict = [ds.as_dict() for ds in bd.Database('ecoinvent-3.10-cutoff-' + premise_db_name)]

    db_linked = copy.deepcopy(base_db)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in db_linked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchange_link = wurst.get_one(base_db + premise_db_dict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchange_link['database'], exchange_link['code'])})
                exchange.update({'database' : exchange_link['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchange_link = [ds['code'] for ef in biosphere_db_dict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchange_link)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    db_name = 'acetylene-production-' + premise_db_name
    if db_name in bd.databases:
        del bd.databases[db_name]
    wurst.write_brightway2_database(db_linked, db_name)
    print(premise_db_name + ' linking and writing successful!')

linking database Base-2050...
29 datasets
297 exchanges
0 unlinked exchanges
  


100%|██████████| 29/29 [00:00<00:00, 2058.78it/s]

Vacuuming database 


Created database: acetylene-production-Base-2050
Base-2050 linking and writing successful!
linking database RCP19-2050...
29 datasets
297 exchanges
0 unlinked exchanges
  


100%|██████████| 29/29 [00:00<00:00, 2058.68it/s]

Vacuuming database 


Created database: acetylene-production-RCP19-2050
RCP19-2050 linking and writing successful!
linking database RCP26-2050...
29 datasets
297 exchanges
0 unlinked exchanges
  


100%|██████████| 29/29 [00:00<00:00, 1131.78it/s]

Vacuuming database 


Created database: acetylene-production-RCP26-2050
RCP26-2050 linking and writing successful!
